# 01 — Exploratory Data Analysis
## Insurance Churn Prediction

This notebook explores the insurance churn dataset to understand:
- Class distribution (churn vs. retained)
- Feature distributions and correlations
- Key actuarial drivers of churn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

DATA_PATH = Path("../data/churn_dataset.parquet")

In [ ]:
df = pd.read_parquet(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
df.info()
df.describe()

## 1. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df["churn_label"].value_counts()
axes[0].bar(["Retained", "Churned"], churn_counts.values, color=["steelblue", "coral"])
axes[0].set_title("Churn Distribution")
axes[0].set_ylabel("Count")

axes[1].pie(churn_counts.values, labels=["Retained", "Churned"],
            autopct="%1.1f%%", colors=["steelblue", "coral"])
axes[1].set_title("Churn Rate")

plt.tight_layout()
plt.show()
print(f"Churn rate: {df['churn_label'].mean():.1%}")

## 2. Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop("churn_label", errors="ignore")

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i, col in enumerate(numeric_cols[:12]):
    ax = axes[i // 4, i % 4]
    for label, color in [(0, "steelblue"), (1, "coral")]:
        subset = df[df["churn_label"] == label][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=f"{'Retained' if label == 0 else 'Churned'}")
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 3. Correlation Matrix

In [ ]:
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 4. Churn Rate by Categorical Features

In [ ]:
cat_cols = ["lob", "channel"]

fig, axes = plt.subplots(1, len(cat_cols), figsize=(12, 4))
for i, col in enumerate(cat_cols):
    churn_by_cat = df.groupby(col)["churn_label"].mean().sort_values(ascending=False)
    axes[i].barh(churn_by_cat.index, churn_by_cat.values, color="coral")
    axes[i].set_xlabel("Churn Rate")
    axes[i].set_title(f"Churn Rate by {col}")
    for j, v in enumerate(churn_by_cat.values):
        axes[i].text(v + 0.005, j, f"{v:.1%}", va="center")

plt.tight_layout()
plt.show()

## 5. Premium vs. Churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, color, name in [(0, "steelblue", "Retained"), (1, "coral", "Churned")]:
    subset = df[df["churn_label"] == label]
    ax.scatter(subset["tenure_months"], subset["annual_premium"],
               alpha=0.3, c=color, label=name, s=10)
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Annual Premium (EUR)")
ax.set_title("Premium vs. Tenure by Churn Status")
ax.legend()
plt.tight_layout()
plt.show()

## Key Findings

- **Churn rate**: ~15-20% (imbalanced classification problem)
- **Premium**: Churners tend to have higher premiums relative to their LOB
- **Tenure**: New customers (< 12 months) churn significantly more
- **Claims**: Negative claims experience correlates with churn
- **Channel**: Online customers show higher price sensitivity